# Mars landform classifier (DoMars16k)

Trains a model that looks at an orbital image of Mars and says which landform it shows
(dunes, crater, channel, gullies, ...). This is step 1 of the "Mars Among Us" pipeline:
landform → matching Earth analog sites.

**Before you run (Kaggle):**
1. Right panel → **Session options** → **Accelerator: GPU T4 x2** (or P100).
2. Right panel → **Internet: On** (needs a phone-verified Kaggle account).
3. Top menu → **Run All**. Takes about 20–40 minutes.

**Data:** DoMars16k — 16,150 CTX images, 15 landforms, CC-BY-4.0.
Wilhelm et al., *Remote Sensing* 2020, 12, 3981. https://doi.org/10.5281/zenodo.4291940

In [ ]:
import hashlib, json, random, time, zipfile
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

try:
    import timm
except ImportError:
    %pip install -q timm
    import timm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
WORK = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
print('Device:', DEVICE, torch.cuda.get_device_name(0) if DEVICE == 'cuda' else '(no GPU: turn it on in Session options)')

## 1. Download the dataset (128 MB) and verify it

In [ ]:
URL = 'https://zenodo.org/api/records/4291940/files/data.zip/content'
MD5 = '6457e1fa5a685af059ed939ee2ec3eea'
zip_path, data_root = WORK / 'domars16k.zip', WORK / 'domars16k'

if not data_root.exists():
    import urllib.request
    urllib.request.urlretrieve(URL, zip_path)
    digest = hashlib.md5(zip_path.read_bytes()).hexdigest()
    assert digest == MD5, f'Download corrupted: md5 {digest}'
    zipfile.ZipFile(zip_path).extractall(data_root)
    zip_path.unlink()

# The zip holds train/val/test folders, each with one sub-folder per landform.
def find_split(name):
    hits = [p for p in data_root.rglob(name) if p.is_dir() and any(p.iterdir())]
    assert hits, f'No "{name}" folder found under {data_root}'
    return hits[0]

SPLITS = {s: find_split(s) for s in ('train', 'val', 'test')}
for s, p in SPLITS.items():
    print(s, p, sum(1 for _ in p.rglob('*.*')), 'images')

## 2. Data pipeline
Orbital images have no 'up', so random 90° rotations and flips are valid augmentations.

In [ ]:
# Mapping from the paper's Table 1 (the Zenodo page swaps aec/ael).
CLASS_NAMES = {
    'aec': 'Aeolian Curved', 'ael': 'Aeolian Straight', 'cli': 'Cliff', 'rid': 'Ridge',
    'fsf': 'Channel', 'sfe': 'Mounds', 'fsg': 'Gullies', 'fse': 'Slope Streaks',
    'fss': 'Mass Wasting', 'cra': 'Crater', 'sfx': 'Crater Field', 'mix': 'Mixed Terrain',
    'rou': 'Rough Terrain', 'smo': 'Smooth Terrain', 'tex': 'Textured Terrain',
}
IMG = 224
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

class RandomRot90:
    def __call__(self, x):
        return torch.rot90(x, random.randint(0, 3), dims=(1, 2))

train_tf = transforms.Compose([
    transforms.Grayscale(3),
    transforms.RandomResizedCrop(IMG, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(), RandomRot90(), transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    transforms.Grayscale(3), transforms.Resize(IMG),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD),
])

train_ds = datasets.ImageFolder(SPLITS['train'], train_tf)
val_ds = datasets.ImageFolder(SPLITS['val'], eval_tf)
test_ds = datasets.ImageFolder(SPLITS['test'], eval_tf)
assert train_ds.classes == val_ds.classes == test_ds.classes
CLASSES = train_ds.classes
print(len(CLASSES), 'classes:', [CLASS_NAMES.get(c, c) for c in CLASSES])

loader = lambda ds, shuffle: DataLoader(ds, batch_size=64, shuffle=shuffle, num_workers=2, pin_memory=True)
train_dl, val_dl, test_dl = loader(train_ds, True), loader(val_ds, False), loader(test_ds, False)

## 3. Fine-tune a pretrained model
We start from ConvNeXt-Nano (pretrained on ImageNet) instead of training from scratch: much higher accuracy in far less time.

In [ ]:
EPOCHS = 12
# ConvNeXt-Nano: small enough that the int8 model fits GitHub's 25 MB browser-upload limit.
MODEL_NAME = 'convnext_nano.in12k_ft_in1k'
model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=len(CLASSES)).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.05)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=1e-4, total_steps=EPOCHS * len(train_dl), pct_start=0.1)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
scaler = torch.amp.GradScaler(enabled=DEVICE == 'cuda')

@torch.no_grad()
def predict(dl, tta=False):
    model.eval()
    probs, labels = [], []
    for x, y in dl:
        x = x.to(DEVICE)
        with torch.autocast(DEVICE, enabled=DEVICE == 'cuda'):
            views = [x] if not tta else [torch.rot90(v, k, dims=(2, 3)) for v in (x, x.flip(3)) for k in range(4)]
            p = torch.stack([model(v).float().softmax(1) for v in views]).mean(0)
        probs.append(p.cpu()); labels.append(y)
    return torch.cat(probs), torch.cat(labels)

best_acc, best_path = 0.0, WORK / 'mars_landforms_best.pt'
for epoch in range(1, EPOCHS + 1):
    model.train(); start, total_loss = time.time(), 0.0
    for x, y in train_dl:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(DEVICE, enabled=DEVICE == 'cuda'):
            loss = criterion(model(x), y)
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update(); scheduler.step()
        total_loss += loss.item() * len(y)
    probs, labels = predict(val_dl)
    acc = (probs.argmax(1) == labels).float().mean().item()
    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), best_path)
    print(f'epoch {epoch:2d}  loss {total_loss / len(train_ds):.3f}  val acc {acc:.2%}  best {best_acc:.2%}  ({time.time() - start:.0f}s)')

## 4. Final score on the test set (images the model never saw)
This is the number you show the judges.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

model.load_state_dict(torch.load(best_path, map_location=DEVICE))
probs, labels = predict(test_dl, tta=True)
preds = probs.argmax(1)
test_acc = (preds == labels).float().mean().item()
names = [CLASS_NAMES.get(c, c) for c in CLASSES]
print(f'TEST ACCURACY: {test_acc:.2%} on {len(labels)} images\n')
print(classification_report(labels, preds, target_names=names, digits=3, zero_division=0))

cm = confusion_matrix(labels, preds, normalize='true')
fig, ax = plt.subplots(figsize=(9, 8))
ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(names)), names, rotation=90); ax.set_yticks(range(len(names)), names)
ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title(f'Test accuracy {test_acc:.1%}')
plt.tight_layout(); plt.savefig(WORK / 'confusion_matrix.png', dpi=120); plt.show()

## 5. Save everything for the website
Download these from the **Output** panel: `mars_landforms.int8.onnx` (the website model), `labels.json`, `confusion_matrix.png`. Keep `mars_landforms_best.pt` as a backup.

In [ ]:
try:
    import onnx, onnxscript, onnxruntime  # not all preinstalled on Kaggle
except ImportError:
    %pip install -q onnx onnxscript onnxruntime
import onnxruntime as ort
from onnxruntime.quantization import QuantType, quantize_dynamic

fp32_path, int8_path = WORK / 'mars_landforms_fp32.onnx', WORK / 'mars_landforms.int8.onnx'
model.eval().cpu()
onnx_args = dict(input_names=['image'], output_names=['logits'])
try:
    torch.onnx.export(model, (torch.randn(1, 3, IMG, IMG),), fp32_path, external_data=False, **onnx_args)
except TypeError:  # older PyTorch without the new exporter options
    torch.onnx.export(model, (torch.randn(1, 3, IMG, IMG),), fp32_path, opset_version=17, **onnx_args)

# int8 weights (~4x smaller) so the model loads fast in the browser via onnxruntime-web.
# Drop the exporter's cached shapes first: they trip the quantizer's shape inference.
graph_model = onnx.load(fp32_path)
del graph_model.graph.value_info[:]
onnx.save(graph_model, fp32_path)
quantize_dynamic(fp32_path, int8_path, weight_type=QuantType.QInt8, op_types_to_quantize=['Conv', 'MatMul', 'Gemm'])

# Score the int8 file itself on the test set: this is the model the website runs.
session = ort.InferenceSession(str(int8_path), providers=['CPUExecutionProvider'])
int8_preds = [session.run(None, {'image': x.unsqueeze(0).numpy()})[0].argmax() for x, _ in test_ds]
int8_acc = float(np.mean(np.array(int8_preds) == np.array(test_ds.targets)))
size_mb = lambda p: p.stat().st_size / 1e6
print(f'fp32 ONNX {size_mb(fp32_path):.1f} MB | int8 ONNX {size_mb(int8_path):.1f} MB')
print(f'TEST ACCURACY  PyTorch: {test_acc:.2%}  int8 ONNX: {int8_acc:.2%}')

meta = {
    'model': MODEL_NAME, 'classes': CLASSES, 'names': names, 'image_size': IMG, 'mean': MEAN, 'std': STD,
    'grayscale': True, 'test_accuracy': round(test_acc, 4), 'int8_test_accuracy': round(int8_acc, 4),
    'test_images': len(labels), 'dataset': 'DoMars16k (doi:10.5281/zenodo.4291940, CC-BY-4.0)',
}
(WORK / 'labels.json').write_text(json.dumps(meta, indent=2))
print(json.dumps(meta, indent=2))
